## Setting the ticker data and imports

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, LongType
import requests
import time

POLYGON_API_KEY = dbutils.secrets.get(scope="polygon_sm", key="polygon_api_key")

TICKERS = ["AAPL","MSFT","GOOGL","AMZN","TSLA","META","NVDA","JPM","V","UNH","XOM","JNJ","WMT","NFLX","MA"]

NUM_DAYS = 365
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=NUM_DAYS)

#Volume - Landing Zone
VOLUME_LANDING = "/Volumes/tabular/dataexpert/sm_capstone_stocks/landing/raw/"

POLYGON_SCHEMA = StructType([
    StructField("v",  DoubleType(), True),
    StructField("vw", DoubleType(), True),
    StructField("o",  DoubleType(), True),
    StructField("c",  DoubleType(), True),
    StructField("h",  DoubleType(), True),
    StructField("l",  DoubleType(), True),
    StructField("t",  LongType(),   True),
    StructField("n",  LongType(),   True),
])

print(f"Analyzing {TICKERS} from {START_DATE:%Y-%m-%d} to {END_DATE:%Y-%m-%d} ({NUM_DAYS} days, 1-min bars)")


## Fetching data from API

In [0]:
def fetch_polygon_bars(ticker, start_date, end_date, api_key, timespan="minute", multiplier=1):
    """Fetch OHLCV bars from Polygon.io REST API with automatic pagination."""
    all_results = []
    url = (
        f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/"
        f"{start_date:%Y-%m-%d}/{end_date:%Y-%m-%d}"
        f"?adjusted=true&sort=asc&limit=50000&apiKey={api_key}"
    )
    page = 1
    while url:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        print(f"Fetched data from page {page}. Waiting 12 seconds to avoid rate limits...")
        time.sleep(12)
        try:
            data = resp.json()
        except ValueError:
            print(f"  Warning: malformed JSON on page {page} for {ticker}, retrying...")
            time.sleep(12)
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            data = resp.json()
        results = data.get("results", [])
        all_results.extend(results)
        next_url = data.get("next_url")
        if next_url:
            url = f"{next_url}&apiKey={api_key}"
            page += 1
        else:
            url = None
    print(f"  Fetched {len(all_results):,} {timespan} bars for {ticker} ({page} page(s))")
    return all_results


def to_minute_df(results, ticker, schema):
    """Convert Polygon API results to a Spark DataFrame with ticker, timestamp, and trade_date."""
    raw_df = spark.createDataFrame(results, schema=schema)
    return (
        raw_df
        .withColumn("ticker", F.lit(ticker))
        .withColumn("timestamp", F.to_timestamp(F.col("t") / 1000))
        .withColumn("trade_date", F.to_date(F.col("timestamp")))
        .withColumn("fetched_at", F.current_timestamp())
        .select(
            F.col("ticker"),
            F.col("timestamp"),
            F.col("trade_date"),
            F.col("o").alias("open"),
            F.col("h").alias("high"),
            F.col("l").alias("low"),
            F.col("c").alias("close"),
            F.col("v").alias("volume"),
            F.col("vw").alias("vwap"),
            F.col("n").alias("transactions"),
            "fetched_at"
        )
    )

for ticker in TICKERS:
    print(f"Processing {ticker}...")
    minute_results = fetch_polygon_bars(ticker, START_DATE, END_DATE, POLYGON_API_KEY)
    minute_df = to_minute_df(minute_results, ticker, POLYGON_SCHEMA)
    
    # Write to Volume as Parquet, partitioned by trade_date for efficient reads
    out_path = f"{VOLUME_LANDING}/{ticker}"
    minute_df.write.mode("overwrite").partitionBy("trade_date").parquet(out_path)
    print(f"Wrote {minute_df.count():,} rows to {out_path}")